In [1]:
!pip install gradio reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.0 MB/s eta 0:00:00


In [2]:
import ast
import gradio as gr
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

In [3]:
class DocGenieAnalyzer:

    def extract_function_signature(self, code):
        tree = ast.parse(code)

        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):

                name = node.name
                params = []

                for arg in node.args.args:
                    params.append({
                        "name": arg.arg,
                        "type": "Any"
                    })

                return {
                    "name": name,
                    "params": params,
                    "return_type": "Any"
                }

        return None


    def analyze_function_logic(self, code):
        tree = ast.parse(code)

        has_loop = False
        has_condition = False

        for node in ast.walk(tree):

            if isinstance(node, (ast.For, ast.While)):
                has_loop = True

            if isinstance(node, ast.If):
                has_condition = True

        description = "Performs computation"

        if has_loop:
            description += " using loops"

        if has_condition:
            description += " with conditional logic"

        return description

In [4]:
def generate_google_doc(signature, description):

    doc = '"""\n'
    doc += description + ".\n\n"

    doc += "Args:\n"

    for p in signature["params"]:
        doc += f'    {p["name"]} (Any): parameter.\n'

    doc += "\nReturns:\n"
    doc += "    Any: result.\n"

    doc += '"""'

    return doc

In [5]:
def generate_numpy_doc(signature, description):

    doc = '"""\n'
    doc += description + ".\n\n"

    doc += "Parameters\n----------\n"

    for p in signature["params"]:
        doc += f'{p["name"]} : Any\n'
        doc += "    parameter.\n"

    doc += "\nReturns\n-------\n"
    doc += "Any\n"
    doc += "    result.\n"

    doc += '"""'

    return doc

In [6]:
analyzer = DocGenieAnalyzer()

def generate_docstring(code, style):

    signature = analyzer.extract_function_signature(code)
    description = analyzer.analyze_function_logic(code)

    if style == "google":
        doc = generate_google_doc(signature, description)
    else:
        doc = generate_numpy_doc(signature, description)

    lines = code.split("\n")

    if len(lines) > 1:
        lines.insert(1, "    " + doc)

    return "\n".join(lines)

In [7]:
def export_pdf(text):

    filename = "docgenie_output.pdf"

    styles = getSampleStyleSheet()

    story = []
    story.append(Paragraph("Doc-Genie Generated Documentation", styles['Title']))
    story.append(Spacer(1,20))
    story.append(Paragraph(text.replace("\n","<br/>"), styles['BodyText']))

    pdf = SimpleDocTemplate(filename)
    pdf.build(story)

    return filename

In [8]:
with gr.Blocks() as demo:

    gr.Markdown("# Doc-Genie Python Docstring Generator")

    code_input = gr.Textbox(
        label="Enter Python Function",
        lines=10,
        value="""def factorial(n):
    if n==0:
        return 1
    return n*factorial(n-1)"""
    )

    style = gr.Radio(
        ["google","numpy"],
        value="google",
        label="Docstring Style"
    )

    generate_btn = gr.Button("Generate Docstring")

    output = gr.Code(label="Generated Code")

    pdf_btn = gr.Button("Download PDF")

    pdf_file = gr.File()

    generate_btn.click(
        generate_docstring,
        inputs=[code_input,style],
        outputs=output
    )

    pdf_btn.click(
        export_pdf,
        inputs=output,
        outputs=pdf_file
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://445e66fff9e48c9d63.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
